# 2 — Lead Scoring Modeli (LR baseline + LightGBM)

> **Yazılı karşılığı:** [`docs/2_lead_scoring.docx`](../docs/2_lead_scoring.docx) — tüm metodoloji, model seçim gerekçeleri, kalibrasyon ve lift yorumu docx'te. Bu notebook sayıları + grafikleri canlı koşturur ve docx'te raporlanan değerleri yeniden üretir.

Bu defter, `scripts/train_lead_scoring.py` tarafından eğitilmiş Logistic Regression ve LightGBM modellerini yükleyip değerlendirme metriklerini (ROC, PR, kalibrasyon, lift, feature importance) görselleştirir. Eğitim burada tekrar koşturulmaz; canonical artefaktlar `artifacts/lead_scoring_*.joblib` üzerinden okunur.

## 1. Kurulum ve veri yükleme

Phase 1'in fitted feature pipeline'ı (`FeatureTransformer.load`) ham CSV'yi 104-boyutlu sayısal matrise dönüştürür. CV foldları içinde yeniden fit edilmez — clip yüzdeliği, OHE kategorileri ve scaler istatistikleri eğitim setinde bir kez kalibre edilmiştir. LightGBM'in JSON-özel karakterleri kabul etmemesi nedeniyle bazı OHE kolon isimleri training script'inde sanitize edilmiştir; aynı sanitize listesi modellerin `feature_names` alanında saklıdır.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split

from lead_priority.features import FeatureTransformer
from lead_priority.features.constants import SEED
from lead_priority.models import LeadScoringModel
from lead_priority.settings import REPO_ROOT

RAW_CSV = REPO_ROOT / 'data' / 'Lead Scoring.csv'
PIPELINE_PATH = REPO_ROOT / 'artifacts' / 'feature_pipeline.joblib'
LR_PATH = REPO_ROOT / 'artifacts' / 'lead_scoring_lr.joblib'
LGBM_PATH = REPO_ROOT / 'artifacts' / 'lead_scoring_lgbm.joblib'
METRICS_PATH = REPO_ROOT / 'artifacts' / 'lead_scoring_metrics.json'

raw = pd.read_csv(RAW_CSV)
transformer = FeatureTransformer.load(PIPELINE_PATH)
lr_model = LeadScoringModel.load(LR_PATH)
lgbm_model = LeadScoringModel.load(LGBM_PATH)

feature_names = list(lr_model.feature_names)
x = pd.DataFrame(transformer.transform(raw), columns=feature_names)
y = raw['Converted'].to_numpy().astype(int)
print(f'feature matrix: {x.shape} | positive rate: {y.mean():.4f}')

## 2. Stratified 60/20/20 ayrım

`Converted` üzerinde stratified split: önce test (%20) ayrılır, ardından kalanın %25'i (toplamda %20) validation olarak ayrılır. `random_state=42` (Phase 0 + Phase 1 ile tutarlı) seed sabittir, böylece bu defter `scripts/train_lead_scoring.py`'in ürettiği aynı bölümleri görür ve metrikler birebir eşleşir.

In [ ]:
x_trainval, x_test, y_trainval, y_test = train_test_split(
    x, y, test_size=0.20, stratify=y, random_state=SEED
)
x_train, x_val, y_train, y_val = train_test_split(
    x_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=SEED
)
print(f'train={len(y_train)}, val={len(y_val)}, test={len(y_test)}')
print(f'positive rate per split: train={y_train.mean():.4f}, val={y_val.mean():.4f}, test={y_test.mean():.4f}')

## 3. Train / val / test metrikleri

İki ana metrik: **ROC-AUC** modelin rastgele bir pozitif örneği rastgele bir negatife göre daha yüksek skorlama olasılığını verir (1.0 mükemmel, 0.5 rastgele); **PR-AUC** (Average Precision) dengesiz sınıflarda precision/recall trade-off'unun bir özeti olarak ROC'a göre daha sıkı bir performans göstergesidir. Aşağıdaki tablo `artifacts/lead_scoring_metrics.json`'dan okunur — yani training script'in ürettiği canonical sayılarla aynı.

In [ ]:
metrics = json.loads(METRICS_PATH.read_text())
rows = []
for kind, summary in metrics['models'].items():
    for split in ('train', 'val', 'test'):
        rows.append({
            'model': kind,
            'split': split,
            'roc_auc': summary['roc_auc'][split],
            'pr_auc': summary['pr_auc'][split],
        })
metric_table = pd.DataFrame(rows).pivot(index='model', columns='split', values=['roc_auc', 'pr_auc'])
metric_table.round(4)

In [ ]:
for kind, summary in metrics['models'].items():
    print(f'{kind:22s} best params: {summary["best_params"]}')

## 3.1 Bootstrap 95% güven aralığı (marginal, test seti)

Test seti tek bir örneklemden ibarettir; raporlanan AUC nokta tahminleri başka bir test split'inde birebir aynı kalmaz. Stratified bootstrap (1.000 yeniden örnekleme, seed=42) test setinin sampling varyansını ölçer ve her metriğin etrafına 95% güven aralığı yerleştirir. Bu tablo her modeli **bağımsız** değerlendirir — örtüşmeyen CI'lar güçlü bir anlamlılık göstergesidir; örtüşen CI'lar ise modeller arası farkın yokluğunu **garantilemez**, çünkü iki model AYNI satırlar üzerinde test edildi (bunu yakalamak için §3.2'deki paired test).

In [ ]:
ci_rows = []
for kind, summary in metrics['models'].items():
    ci = summary['test_ci_95']
    ci_rows.append({
        'model': kind,
        'test ROC-AUC': summary['roc_auc']['test'],
        'ROC-AUC CI lower': ci['roc_auc']['lower'],
        'ROC-AUC CI upper': ci['roc_auc']['upper'],
        'test PR-AUC': summary['pr_auc']['test'],
        'PR-AUC CI lower': ci['pr_auc']['lower'],
        'PR-AUC CI upper': ci['pr_auc']['upper'],
    })
ci_table = pd.DataFrame(ci_rows).set_index('model').round(4)
ci_table

## 3.2 Paired bootstrap — LGBM vs. LR farkının CI'sı

Marginal CI'ların örtüşmesi yanıltıcı olabilir. İki model AYNI test satırlarında değerlendirildiğine göre, doğru test her resample'da (LGBM AUC − LR AUC) farkının CI'sına bakmaktır. Eğer CI **0'ı içermiyorsa** üstünlük per-sample tutarlıdır ve marginal CI'ların örtüştüğü durumlarda dahi istatistiksel olarak anlamlı sayılır.

In [ ]:
paired = metrics['model_comparison']['paired_roc_auc_diff_lgbm_minus_lr']
print(f'LGBM − LR test ROC-AUC paired diff (mean): {paired["point_estimate"]:.4f}')
print(f'95% CI: [{paired["lower"]:.4f}, {paired["upper"]:.4f}]')
print(f'bootstrap iterations: {paired["n_iters"]}')
print(f'CI excludes 0? {paired["significant_at_95"]}  →  ' +
      ('LGBM üstünlüğü 95% güven seviyesinde istatistiksel olarak ANLAMLI.'
       if paired['significant_at_95']
       else 'fark istatistiksel olarak ANLAMLI bulunmadı.'))

## 4. ROC ve PR eğrileri

Test seti üzerinde iki model üst üste çizilir. ROC eğrisi false-positive rate vs. true-positive rate; PR eğrisi recall vs. precision'ı gösterir. PR eğrisinin altındaki alan, pozitif sınıf görece azaldıkça (%38.5 burada) ROC-AUC'tan daha sıkı bir gösterge olarak okunur.

In [ ]:
proba_lr_test = lr_model.predict_proba(x_test.to_numpy())
proba_lgbm_test = lgbm_model.predict_proba(x_test.to_numpy())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for name, proba, color in [
    ('LR', proba_lr_test, 'tab:blue'),
    ('LGBM', proba_lgbm_test, 'tab:orange'),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, color=color, label=f'{name} (AUC={auc:.3f})')
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[1].plot(rec, prec, color=color, label=f'{name} (AP={ap:.3f})')

axes[0].plot([0, 1], [0, 1], linestyle='--', color='grey', linewidth=0.8)
axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate')
axes[0].set_title('ROC — test set'); axes[0].legend(loc='lower right'); axes[0].grid(alpha=0.3)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall — test set'); axes[1].legend(loc='lower left'); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Kalibrasyon eğrisi (reliability diagram)

Olasılık çıktısının ne kadar güvenilir olduğunu ölçer. Test örnekleri tahmin olasılıklarına göre 10 quantile-bucket'a ayrılır; her bucket için ortalama tahmin (x ekseni) ve gerçek pozitif oranı (y ekseni) karşılaştırılır. İdeal kalibrasyon `y = x` köşegeni üzerindedir; altında kalan eğri "under-confident" (modelin tahmini gerçekten daha düşük), üstünde olan eğri "over-confident" anlamına gelir.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for name, proba, color in [
    ('LR', proba_lr_test, 'tab:blue'),
    ('LGBM', proba_lgbm_test, 'tab:orange'),
]:
    frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10, strategy='quantile')
    ax.plot(mean_pred, frac_pos, marker='o', color=color, label=name)
ax.plot([0, 1], [0, 1], linestyle='--', color='grey', linewidth=0.8, label='ideal')
ax.set_xlabel('Ortalama tahmin (quantile bucket)')
ax.set_ylabel('Gerçek pozitif oranı')
ax.set_title('Reliability diagram — test set (10 bucket)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Lift / cumulative gain analizi

Satış ekibi için anlamlı tek soru: "Modelin sıralamasında üst %X'lik dilimi seçersem, dönüşümlerin yüzde kaçını yakalarım?" Aşağıdaki kümülatif kazanım eğrisi bu soruyu doğrudan cevaplar. %20 noktasındaki değer, leads zamanı kısıtlı bir ekibin somut prioritization kazancını gösterir.

In [ ]:
def cumulative_gain_curve(y_true, y_proba):
    order = np.argsort(-y_proba)
    cum_pos = np.cumsum(y_true[order])
    total_pos = cum_pos[-1]
    fractions = np.arange(1, len(y_true) + 1) / len(y_true)
    gains = cum_pos / total_pos
    return fractions, gains

fig, ax = plt.subplots(figsize=(7, 6))
for name, proba, color in [
    ('LR', proba_lr_test, 'tab:blue'),
    ('LGBM', proba_lgbm_test, 'tab:orange'),
]:
    fracs, gains = cumulative_gain_curve(y_test, proba)
    ax.plot(fracs, gains, color=color, label=name)
ax.plot([0, 1], [0, 1], linestyle='--', color='grey', linewidth=0.8, label='rastgele')
for x_mark in (0.10, 0.20, 0.30):
    ax.axvline(x_mark, linestyle=':', color='grey', linewidth=0.6)
ax.set_xlabel('Sıralanmış lead havuzu (top fraction)')
ax.set_ylabel('Yakalanan dönüşüm oranı')
ax.set_title('Cumulative gain — test set')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

lift_rows = []
for kind, summary in metrics['models'].items():
    row = {'model': kind}
    row.update({f'top_{k}%': v for k, v in zip(('10', '20', '30'), summary['lift'].values())})
    lift_rows.append(row)
pd.DataFrame(lift_rows).set_index('model').round(4)

## 7. Karar eşiği taraması ve operasyonel öneriler

Cumulative gain "üst %X" perspektifini verir; gerçek uygulamada satış ekibi olasılık eşiği üzerinden karar verir. Aşağıdaki tablo {0.3, 0.4, 0.5, 0.6, 0.7} aralığındaki her eşik için: kaç lead'in yüksek-öncelik olarak işaretlendiğini (`predicted_positive_rate`), bu işaretlemenin precision'unu (yanlış pozitif oranı) ve recall'unu (gerçek dönüşümün yakalanma oranı) verir.

Pratikte iki ana yaklaşım vardır: **(a) sabit kapasite** — ekip günde N lead arayabiliyorsa, model sıralamasında üst N seçilir, eşik dinamik olarak kapasiteye göre belirlenir. **(b) sabit precision** — kabul edilebilir yanlış pozitif oranı tavanı verilir (örn. %25), modelden bu precision'a karşılık gelen eşik geriye doğru hesaplanır.

In [ ]:
sweep_rows = []
for kind, summary in metrics['models'].items():
    for row in summary['threshold_sweep']:
        sweep_rows.append({'model': kind, **row})
sweep_df = pd.DataFrame(sweep_rows)
sweep_table = sweep_df.pivot(
    index='threshold',
    columns='model',
    values=['predicted_positive_rate', 'precision', 'recall'],
)
sweep_table.round(3)

## 8. LightGBM feature importance (gain)

Modelin split anlarında elde ettiği toplam bilgi kazancına göre en önemli 15 feature. `importance_type='split'` (varsayılan split sayısı) yerine `gain` kullanılır: gain doğrudan modelin loss düşüşünü mevcut kolona atfeder, dolayısıyla iş kararı yorumuna daha yakındır. Bu görsel modelin **genel** davranışını anlatır; tekil bir lead için "neden bu skor?" sorusu §9'daki SHAP analiziyle yanıtlanır.

In [ ]:
booster = lgbm_model.classifier.booster_
gain = booster.feature_importance(importance_type='gain')
names = booster.feature_name()
importance = pd.Series(gain, index=names).sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(8, 6))
importance.plot(kind='barh', ax=ax, color='tab:orange')
ax.set_xlabel('Gain importance')
ax.set_title('LightGBM — top 15 feature (gain)')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 9. Tekil lead için açıklama (SHAP)

Feature importance modelin **genel** davranışını anlatır; satış temsilcisi tek bir lead için "bu lead neden bu kadar yüksek skor aldı?" sorusunu sorduğunda tek başına yetmez. SHAP (SHapley Additive exPlanations) her feature'ın o tekil tahmine katkısını oyun teorisi tabanlı bir adil-paylaşım çerçevesinde hesaplar.

Aşağıda test setinden yüksek-skorlu bir lead için **waterfall plot** çizilir: tahmin, base value'dan (`expected_value`, modelin ortalama log-odds çıktısı) başlayarak feature katkılarıyla nihai skora ulaşılır. Her bar bir feature'ın o lead'in skoruna katkısını gösterir; pozitif (kırmızı) skor artırır, negatif (mavi) düşürür.

İkinci grafik **summary bar** plot'tur: test örneklerinin tamamı üzerinde her feature'ın ortalama mutlak SHAP değerini (|SHAP|) gösterir; gain importance'a paralel ama her örneğin katkısı eşit ağırlıkla toplandığı için yorumlaması daha doğrudandır.

In [ ]:
import shap

# SHAP'i 200 satırlık bir test örneklemi üzerinde hesapla (waterfall için tam test
# seti gereksiz büyük; summary plot için 200 satır temsil için yeterli).
sample = x_test.head(200).reset_index(drop=True)
sample_proba = lgbm_model.predict_proba(sample.to_numpy())

explainer = shap.TreeExplainer(lgbm_model.classifier)
shap_values = explainer(sample)

# Açıklamak için yüksek-skorlu bir lead seç (örneklem içindeki argmax)
high_score_idx = int(np.argmax(sample_proba))
print(f'Açıklanan örnek: örneklem içinde {high_score_idx}. satır')
print(f'LGBM tahmin olasılığı: {sample_proba[high_score_idx]:.4f}')
print(f'Gerçek etiket: {y_test.iloc[high_score_idx] if hasattr(y_test, "iloc") else y_test[high_score_idx]}')

shap.plots.waterfall(shap_values[high_score_idx], max_display=10, show=True)

In [ ]:
shap.plots.bar(shap_values, max_display=10, show=True)

## 10. Sonuç ve Phase 5 entegrasyonu

Test seti üzerinde LightGBM, LR baseline'ına göre ROC-AUC ve PR-AUC'ta belirgin bir farkla önde çıkar. Marginal bootstrap CI'ları minik bir aralıkla örtüşür (§3.1), ancak iki model AYNI satırlarda test edildiği için doğru istatistiksel yargı paired bootstrap'tır: (LGBM − LR) ROC-AUC farkının 95% güven aralığı 0'ı içermez (§3.2), dolayısıyla LGBM üstünlüğü istatistiksel olarak anlamlıdır. Lift tablosu üst %20'lik dilimde tek başına dönüşümlerin yarısına yakınını yakalar (§6). Karar eşiği taraması operasyonel karar için iki ana yaklaşımı (sabit kapasite ve sabit precision) somut sayılarla karşılaştırılabilir hâle getirir (§7). Kalibrasyon eğrisi her iki modelin de büyük ölçüde köşegen civarında olduğunu, bu nedenle isotonic / Platt scaling adımının şu an için zorunlu olmadığını gösterir (§5). SHAP analizi tekil bir lead'in skorunun hangi feature katkılarından oluştuğunu satır-bazlı görselleştirir (§9), gain importance'ın global özetini tamamlar.

Yapım sırasının bir sonraki adımı Phase 5 (FastAPI): `LeadScoringModel.load(...)` bu defterde kullanılan aynı API üzerinden çağrılır, `predict_proba` aynı `np.ndarray → np.ndarray` sözleşmesini sağlar. Sentiment sinyali (Phase 3) ve birleşik priority skoru (Phase 4) bu modelin çıktısının üzerine inşa edilir.